# Study 842 — Implementation Shortfall 🧾

**The paper-vs-live cost gap: the same strategy at 0 / realistic / stressed cost.**

André Perold (1988): the frictionless *paper portfolio* and the real portfolio differ by the
cost of trading into the positions — and that gap scales with **turnover**. We take a
moderate-turnover cross-sectional long-short with a **planted, genuine gross edge** (so its
0-cost Sharpe honestly dazzles: **2.27**, NW *t* = **+7.65**) and
evaluate the *identical* book across a cost ladder with a turnover-scaled market-impact term.

*Synthetic-only by design (a real tape can't certify a clean planted edge), so this is a
method demo capped at `NONE`. Numbers below are the frozen headline (`docs/results.md`,
fp `79082a088002`, as-of 2026-06-30); the live cells run the fast synthetic control.*


## 1. The idea in one picture

A backtest buys and sells at the *decision* price and pays nothing to trade — the **paper portfolio**. The real book pays the spread, a commission, and, worst of all, **market impact**: the more of the book you rotate, the more your own trading moves the price against you. The paper return minus the real return is the *implementation shortfall*, and for a strategy that trades a lot it can be the entire edge.

In [1]:
R = dict(gross_sharpe=2.27, gross_t=7.65, turnover=0.348, breakeven=35.24)
print('PAPER portfolio (0 cost): gross Sharpe %.2f  (NW t = %+.2f)'
      % (R['gross_sharpe'], R['gross_t']))
print('  it rotates %.0f%% of the book every day' % (R['turnover']*100))
print('  looks like a real, tradable edge... on paper')

PAPER portfolio (0 cost): gross Sharpe 2.27  (NW t = +7.65)
  it rotates 35% of the book every day
  looks like a real, tradable edge... on paper


## 2. Now charge the cost of trading — the SAME strategy

Zero cost is a fantasy. Add an *optimistic* cost, then a *realistic* one, then a *stressed* one (bigger book, thinner names). The gross Sharpe never changes — it is blind to trading — but the **net** Sharpe tells the truth.

In [2]:
print(f'paper (0 cost)  : gross Sharpe 2.27  ->  net Sharpe   2.27  (net t = +7.65, +30.9%/yr)')
print(f'optimistic      : gross Sharpe 2.27  ->  net Sharpe   1.39  (net t = +4.69, +18.9%/yr)')
print(f'realistic       : gross Sharpe 2.27  ->  net Sharpe   0.24  (net t = +0.79, +3.2%/yr)')
print(f'stressed        : gross Sharpe 2.27  ->  net Sharpe  -1.78  (net t = -5.90, -24.5%/yr)')

paper (0 cost)  : gross Sharpe 2.27  ->  net Sharpe   2.27  (net t = +7.65, +30.9%/yr)
optimistic      : gross Sharpe 2.27  ->  net Sharpe   1.39  (net t = +4.69, +18.9%/yr)
realistic       : gross Sharpe 2.27  ->  net Sharpe   0.24  (net t = +0.79, +3.2%/yr)
stressed        : gross Sharpe 2.27  ->  net Sharpe  -1.78  (net t = -5.90, -24.5%/yr)


## 3. Why it collapses — turnover

The paper alpha is real, but you have to *trade* to capture it, and this book turns over ~35% of NAV a day. At a realistic cost the friction (~11 bps/day) is as big as the gross edge (~12.25 bps/day), so the net edge is gone. Trade *more* and it gets worse — the same paper alpha, evaluated at higher turnover, becomes a disaster:

In [3]:
curve = [(0.995, 0.152, 2.06, 1.4, 72.0), (0.98, 0.254, 2.15, 0.86, 45.9), (0.96, 0.348, 2.27, 0.24, 35.2), (0.9, 0.533, 2.17, -1.73, 22.4), (0.7, 0.893, 2.21, -6.77, 13.8), (0.3, 1.343, 2.46, -15.54, 10.0)]
print('phi   turnover/day  gross Sharpe   net Sharpe (realistic cost)')
for phi, tu, gs, ns, be in curve:
    flag = '  <- tradable' if ns > 0.5 else ('  <- DEAD' if ns < 0 else '')
    print(f'{phi:<5} {tu:>8.3f}      {gs:>6.2f}       {ns:>7.2f}{flag}')

phi   turnover/day  gross Sharpe   net Sharpe (realistic cost)
0.995    0.152        2.06          1.40  <- tradable
0.98     0.254        2.15          0.86  <- tradable
0.96     0.348        2.27          0.24
0.9      0.533        2.17         -1.73  <- DEAD
0.7      0.893        2.21         -6.77  <- DEAD
0.3      1.343        2.46        -15.54  <- DEAD


## 4. Is the paper edge even real? A live synthetic control

We *planted* the gross edge, so it had better show up — and it must vanish on a null world where the signal predicts nothing. Live, offline, a few seconds.

In [4]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from cost_gap import data, strategy as st
null = st.synthetic_detect(data, edge=0.0, n_days=1500)
plant = st.synthetic_detect(data, edge=0.0005, n_days=1500)
print('null world   : gross book NW t = %+.2f  (should be ~0)' % null['gross_t'])
print('planted world: gross book NW t = %+.2f  (should light up)' % plant['gross_t'])

null world   : gross book NW t = +0.23  (should be ~0)
planted world: gross book NW t = +6.71  (should light up)


## 5. The honest verdict

The gross edge is genuinely there — but you cannot afford to trade it. **Signal: None** (a synthetic method demo — a real tape can't certify a planted edge). **Tradability: Mirage** (net Sharpe 2.27 → 0.24 at a realistic cost, -1.78 when stressed). **Does ignoring costs manufacture the edge? Confirmed** — the whole 30.9%/yr paper triumph is the cost of trading that the backtest simply forgot to charge. A backtest without a cost model is meaningless.